# R15-H241 - The engine-matched identity A/B

**Round** R15 - identity-stack shipping. **Author** experiment executor, 2026-07-08.
**Approach** a paired v1/v2 A/B on ONE local engine to escape the Bedrock quota hostage that left H212 INCONCLUSIVE twice.

## Why this run exists

H158 shipped the v2 identity stack (isotonic-calibrated Titan cosine + NLI contradiction veto + logistic
arbitration) behind `resolution.identity_stack: v1|v2`, verdict DEGRADED: both identity clauses passed but recall
dropped, blamed on Bedrock extraction loss. H212 tried to close that confound by re-running the v2 arm under
"quiet" Bedrock and landed INCONCLUSIVE **twice** - the daily-token cap dropped 30 then 50 chunks. The v2-vs-v1
recall delta was always measured against v1's *historical* number extracted on a different day. This run removes the
confound at its root: **both arms fresh, paired, on the local vLLM (gpt-oss-120b) - same engine, same day, same
server** - so the recall delta is engine-clean and quota-free.

## Hypothesis

A paired local-engine A/B (v1 and v2, same 28 docs, gpt-oss-120b, sequential on the idle vLLM) shows v2 recall@16
within 2 points of v1's **same-run** recall (non-regression - the actual H158 question) while reproducing the
identity wins (v2 false merges below v1's by >= 2x, SAME_AS precision proxy >= 0.50) - lifting H158 to CONFIRMED on
a cleaner design than the original.

## Registered clauses and bars

| Clause | Bar | Gates |
| --- | --- | --- |
| **Non-regression** (recall@16) | `recall_v2 - recall_v1 >= -0.02` (within 2 pts of paired v1) | the flip |
| **Identity - false merges** | `v2 false merges <= v1 / 2` (>= 2x fewer) | the flip |
| **Identity - precision proxy** | `v2 SAME_AS precision proxy >= 0.50` | the flip |

**Acceptance bar** - the non-regression clause AND both identity clauses -> H158 CONFIRMED + v2 default flip
approved (GO). Refuted if v2 recall drops > 2 pts against its paired v1 (the stack then owes a mechanism -
forensics on which probes flip).

## Paired discipline and the extraction-variance confound

The `identity_stack` flag changes only RESOLUTION, never extraction. The pairing control is: identical corpus,
engine, config, day and server load - only the flag differs. But the two arms are two SEPARATE ingests (wipe neo4j4
between them, per the registration), and H229 established that temp-0 extraction variance on this model is
**model-inherent (~0.60 mean pairwise Jaccard distance), not serving noise**. So the two arms do NOT see a
byte-identical extraction stream; run-to-run extraction variance is a residual confound reported explicitly here,
not assumed away. Entity/relationship-count deltas between arms conflate (a) that extraction variance with (b) the
intended resolution difference (v2 defers more -> fewer merges -> more surviving entities from the same extraction).

## Instruments (frozen)

- **Identity benchmark** - `reports/identity-benchmark-h101-20260707-094448.json` (298 adjudicated pairs; H101)
- **Recall harness** - `notebooks/h158_measure.py` `precision_proxy` + `recall_at_k`; probe set
  `tests/probes/cpap-probe-set.yml` (24 gold-bearing probes, H34 convention), top_k=16
- **Scratch instance** - neo4j4 `bolt://172.19.0.4:7687` (pinned explicitly per DEF-4/DEF-5)
- **Engine** - local vLLM gpt-oss-120b `http://localhost:8010/v1`, temperature 0.0; Bedrock Titan embeddings only

## GPU Selection

In [1]:
# Measurement here is read-only (vector query + node render + Titan embeddings via Bedrock);
# the v2 NLI veto runs GPU-side during INGEST, not during this measurement. Pin a device before any
# transitive torch import for hygiene (notebook standard).
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")  # RTX 5000 Ada - free tier for light NLI/torch use
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA_VISIBLE_DEVICES = 2


## Imports

In [2]:
import json                                   # artifact read/write
import datetime                                 # UTC stamps
import subprocess                               # driving the ingest / measurement scripts
from pathlib import Path                        # paths
import hashlib                                  # graph fingerprints

from rich import print as rprint                # styled output (no CLI frames)
from rich.table import Table
from rich.console import Console

import sys
sys.path.insert(0, "notebooks")                 # h158_measure lives here
from h158_measure import precision_proxy, recall_at_k  # frozen H158 instruments

console = Console()
rprint("[bold cyan]R15-H241[/bold cyan] imports ready")

2026-07-08 18:00:09.247 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


R15-H241 imports ready

## Configuration

In [3]:
import os; os.chdir("/home/lab/workspace/learning/projects/knowledge-graph-foundry")  # nbconvert runs kernel in notebooks/; anchor to project root
ROOT = Path(".")
FRESH_URI = "bolt://172.19.0.4:7687"            # neo4j4 scratch - MINE (DEF-4/DEF-5)
FRESH_AUTH = ("neo4j", "kgfoundry")
CORPUS = "data/external/cpap-datasheets-and-manuals"   # 28-doc benchmark set
BENCH = "reports/identity-benchmark-h101-20260707-094448.json"  # 298 H101 pairs
PROBES = "tests/probes/cpap-probe-set.yml"      # 24 gold-bearing probes (H34)
TOP_K = 16                                      # recall@16

ARMS = {
    "v1": {"config": "config-h241-v1.yml", "events": "logs/h241-v1-events.jsonl",
           "recall_json": "reports/h241-v1-recall.json", "stack": "v1 (Bayesian posterior)"},
    "v2": {"config": "config-h241-v2.yml", "events": "logs/h241-v2-events.jsonl",
           "recall_json": "reports/h241-v2-recall.json", "stack": "v2 (isotonic cosine + NLI veto + logistic)"},
}

# Registered bars
BAR_RECALL_DELTA = -0.02        # v2 - v1 must be >= this (within 2 pts)
BAR_PRECISION = 0.50            # v2 precision proxy floor
FALSE_MERGE_FACTOR = 2.0        # v2 false merges must be <= v1 / this

STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = f"reports/identity-ab-h241-{STAMP}.json"

t = Table(title="H241 paired A/B configuration", show_header=True, header_style="bold")
t.add_column("arm"); t.add_column("identity_stack"); t.add_column("config"); t.add_column("event log")
for a, m in ARMS.items():
    t.add_row(a, m["stack"], m["config"], m["events"])
console.print(t)
rprint(f"engine [green]local vLLM gpt-oss-120b[/green] @ http://localhost:8010/v1  |  instance [green]{FRESH_URI}[/green]")
rprint(f"corpus [green]{CORPUS}[/green]  |  bench [green]{BENCH}[/green]  |  probes [green]{PROBES}[/green] @ k={TOP_K}")
rprint(f"bars: recall delta >= [yellow]{BAR_RECALL_DELTA}[/yellow]  precision >= [yellow]{BAR_PRECISION}[/yellow]  "
       f"false merges v2 <= v1/[yellow]{FALSE_MERGE_FACTOR:.0f}[/yellow]")
rprint(f"report -> [magenta]{REPORT}[/magenta]  stamp {STAMP}")

                                    H241 paired A/B configuration                                    
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ arm ┃ identity_stack                             ┃ config             ┃ event log                 ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ v1  │ v1 (Bayesian posterior)                    │ config-h241-v1.yml │ logs/h241-v1-events.jsonl │
│ v2  │ v2 (isotonic cosine + NLI veto + logistic) │ config-h241-v2.yml │ logs/h241-v2-events.jsonl │
└─────┴────────────────────────────────────────────┴────────────────────┴───────────────────────────┘

engine local vLLM gpt-oss-120b @ http://localhost:8010/v1  |  instance bolt://172.19.0.4:7687

corpus data/external/cpap-datasheets-and-manuals  |  bench reports/identity-benchmark-h101-20260707-094448.json  | 
probes tests/probes/cpap-probe-set.yml @ k=16

bars: recall delta >= -0.02  precision >= 0.5  false merges v2 <= v1/2

report -> reports/identity-ab-h241-20260708T160009Z.json  stamp 20260708T160009Z

## Preflight - instance + engine pinning, corpus census

Confirms neo4j4 is reachable and pinned, the local engine is serving gpt-oss-120b, and the corpus carries the
expected 28 documents before any arm runs.

In [4]:
import urllib.request
from neo4j import GraphDatabase

# corpus census
docs = sorted(Path(CORPUS).glob("*.pdf"))
rprint(f"corpus documents: [green]{len(docs)}[/green]")
assert len(docs) == 28, f"expected 28 docs, found {len(docs)}"

# engine liveness
with urllib.request.urlopen("http://localhost:8010/v1/models", timeout=10) as r:
    served = json.load(r)["data"][0]["id"]
rprint(f"vLLM serving: [green]{served}[/green]")
assert served == "gpt-oss-120b"

# instance reachability (read-only)
_d = GraphDatabase.driver(FRESH_URI, auth=FRESH_AUTH)
with _d.session() as s:
    n = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
_d.close()
rprint(f"neo4j4 reachable, current entity count: [green]{n}[/green] (state depends on which arm last ran)")

corpus documents: 28

vLLM serving: gpt-oss-120b

neo4j4 reachable, current entity count: 2824 (state depends on which arm last ran)

## Arm execution - the paired ingests

Each arm is one `scripts/h241_arm.sh <arm>` run: wipe neo4j4 -> `kgf init "compare CPAP machines"` -> `kgf ingest`
the 28-doc corpus, all pinned to `config-h241-<arm>.yml`. The ingests are long, so they run as a script under this
notebook's control (progress teed to `logs/h241-ab.log`); the exact commands are recorded below. **v1 recall is
measured and persisted to JSON before the v2 arm wipes neo4j4** - the H158 pattern.

In [5]:
def graph_stats(uri):
    d = GraphDatabase.driver(uri, auth=FRESH_AUTH)
    with d.session() as s:
        e = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
        r = s.run("MATCH ()-[x]->() RETURN count(x) AS c").single()["c"]
        doc = s.run("MATCH (d:KGFDocument) RETURN count(d) AS c").single()["c"]
        ids = [row["id"] for row in s.run("MATCH (e:Entity) RETURN e.id AS id ORDER BY e.id")]
    d.close()
    fp = hashlib.sha256("".join(ids).encode()).hexdigest()[:16]
    return {"entities": e, "relationships": r, "kgf_documents": doc, "entity_id_fingerprint": fp}

def ingest_telemetry(events_path):
    # extraction.warning events = failed/dangling chunks (the H158/H212 telemetry clause)
    warn = 0
    for line in Path(events_path).read_text().splitlines():
        try:
            ev = json.loads(line).get("event", "")
        except json.JSONDecodeError:
            continue
        if ev == "extraction.warning":
            warn += 1
    # token-cap retries must be ZERO on the local engine (the whole point of escaping Bedrock)
    caplog = Path("logs/h241-ab.log")
    cap = caplog.read_text().count("Too many tokens per day") if caplog.exists() else 0
    return {"extraction_warnings": warn, "token_cap_retries": cap}

# The exact commands executed (recorded for provenance; run via scripts/h241_arm.sh):
for a, m in ARMS.items():
    rprint(f"[bold]{a}[/bold]: kgf wipe --yes --config {m['config']}  &&  "
           f"kgf init 'compare CPAP machines' --config {m['config']}  &&  "
           f"kgf ingest {CORPUS} --config {m['config']}")

v1: kgf wipe --yes --config config-h241-v1.yml  &&  kgf init 'compare CPAP machines' --config config-h241-v1.yml  
&&  kgf ingest data/external/cpap-datasheets-and-manuals --config config-h241-v1.yml

v2: kgf wipe --yes --config config-h241-v2.yml  &&  kgf init 'compare CPAP machines' --config config-h241-v2.yml  
&&  kgf ingest data/external/cpap-datasheets-and-manuals --config config-h241-v2.yml

## Per-arm measurement - SAME_AS precision proxy + recall@16

`precision_proxy` reads each arm's event log (files persist, recomputed live here). `recall_at_k` was captured
against each arm's LIVE graph before the next wipe and persisted to JSON. Precision proxy and false merges are
scored against the 298 H101 adjudicated pairs; recall@16 over the 24 gold-bearing probes.

In [6]:
results = {}
for a, m in ARMS.items():
    prec = precision_proxy(m["events"], BENCH)
    rec = json.load(open(m["recall_json"]))
    results[a] = {"precision_proxy": prec, "recall": rec}

t = Table(title="H241 per-arm identity + recall", show_header=True, header_style="bold")
t.add_column("metric"); t.add_column("v1", justify="right"); t.add_column("v2", justify="right")
p1, p2 = results["v1"]["precision_proxy"], results["v2"]["precision_proxy"]
r1, r2 = results["v1"]["recall"], results["v2"]["recall"]
t.add_row("SAME_AS precision proxy", f"{p1['same_as_precision']:.3f}", f"{p2['same_as_precision']:.3f}")
t.add_row("false merges (FP)", str(p1["false_merges"]), str(p2["false_merges"]))
t.add_row("merges / defers / NLI vetoes",
          f"{p1['merges_total']}/{p1['defers_total']}/{p1['nli_vetoes']}",
          f"{p2['merges_total']}/{p2['defers_total']}/{p2['nli_vetoes']}")
t.add_row("recall@16 (mean)", f"{r1['mean_recall']:.4f}", f"{r2['mean_recall']:.4f}")
t.add_row("fully-covered probes", f"{r1['fully_covered']}/{r1['n_probes']}", f"{r2['fully_covered']}/{r2['n_probes']}")
t.add_row("graph entities / rels", f"{r1['entities']}/{r1['relationships']}", f"{r2['entities']}/{r2['relationships']}")
console.print(t)

               H241 per-arm identity + recall               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ metric                       ┃        v1 ┃            v2 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ SAME_AS precision proxy      │     0.083 │         0.611 │
│ false merges (FP)            │        11 │             7 │
│ merges / defers / NLI vetoes │ 525/681/0 │ 1016/1996/355 │
│ recall@16 (mean)             │    0.7708 │        0.7708 │
│ fully-covered probes         │     17/24 │         17/24 │
│ graph entities / rels        │ 2727/6986 │     2824/8534 │
└──────────────────────────────┴───────────┴───────────────┘

## Clause evaluation vs registered bars

In [7]:
recall_delta = r2["mean_recall"] - r1["mean_recall"]
nonreg_ok = recall_delta >= BAR_RECALL_DELTA
fm_ratio = (p1["false_merges"] / p2["false_merges"]) if p2["false_merges"] else float("inf")
fm_ok = p2["false_merges"] <= p1["false_merges"] / FALSE_MERGE_FACTOR
prec_ok = p2["same_as_precision"] >= BAR_PRECISION

go = nonreg_ok and fm_ok and prec_ok
refuted = recall_delta < BAR_RECALL_DELTA

t = Table(title="H241 clause verdicts", show_header=True, header_style="bold")
t.add_column("clause"); t.add_column("measured", justify="right"); t.add_column("bar"); t.add_column("verdict")
t.add_row("non-regression (recall delta)", f"{recall_delta:+.4f}", f">= {BAR_RECALL_DELTA}",
          "[green]PASS[/green]" if nonreg_ok else "[red]FAIL[/red]")
t.add_row("false merges v2 vs v1", f"{p2['false_merges']} vs {p1['false_merges']} ({fm_ratio:.2f}x fewer)",
          f"v2 <= v1/{FALSE_MERGE_FACTOR:.0f}", "[green]PASS[/green]" if fm_ok else "[red]FAIL[/red]")
t.add_row("precision proxy (v2)", f"{p2['same_as_precision']:.3f}", f">= {BAR_PRECISION}",
          "[green]PASS[/green]" if prec_ok else "[red]FAIL[/red]")
console.print(t)

decision = "GO - flip default to v2" if go else ("NO-GO - refuted (recall regression)" if refuted else "NO-GO - a clause failed")
rprint(f"\n[bold]v2 default-flip recommendation:[/bold] "
       f"[{'green' if go else 'red'}]{decision}[/{'green' if go else 'red'}]")

                              H241 clause verdicts                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━┓
┃ clause                        ┃              measured ┃ bar        ┃ verdict ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━┩
│ non-regression (recall delta) │               +0.0000 │ >= -0.02   │ PASS    │
│ false merges v2 vs v1         │ 7 vs 11 (1.57x fewer) │ v2 <= v1/2 │ FAIL    │
│ precision proxy (v2)          │                 0.611 │ >= 0.5     │ PASS    │
└───────────────────────────────┴───────────────────────┴────────────┴─────────┘

v2 default-flip recommendation: NO-GO - a clause failed

## Extraction-variance confound (per H229)

The two arms are separate ingests; temp-0 extraction variance on gpt-oss-120b is model-inherent (~0.60 JD, H229).
Counts and telemetry below quantify the residual confound: warnings must be near-symmetric and token-cap retries
must be ZERO (the local engine has no daily cap - the entire reason for this design).

In [8]:
tel = {a: ingest_telemetry(m["events"]) for a, m in ARMS.items()}
t = Table(title="H241 extraction telemetry (confound magnitude)", show_header=True, header_style="bold")
t.add_column("arm"); t.add_column("entities", justify="right"); t.add_column("rels", justify="right")
t.add_column("extraction warnings", justify="right"); t.add_column("token-cap retries", justify="right")
for a in ARMS:
    r = results[a]["recall"]
    t.add_row(a, str(r["entities"]), str(r["relationships"]),
              str(tel[a]["extraction_warnings"]), str(tel[a]["token_cap_retries"]))
console.print(t)
ent_delta = abs(r2["entities"] - r1["entities"])
rprint(f"entity-count delta between arms: [yellow]{ent_delta}[/yellow] "
       f"(conflates extraction variance with v2's higher defer rate - reported, not assumed away)")
rprint(f"token-cap retries total: [green]{tel['v1']['token_cap_retries'] + tel['v2']['token_cap_retries']}[/green] "
       "(local engine - Bedrock daily cap escaped)")

          H241 extraction telemetry (confound magnitude)           
┏━━━━━┳━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ arm ┃ entities ┃ rels ┃ extraction warnings ┃ token-cap retries ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ v1  │     2727 │ 6986 │                 351 │                 0 │
│ v2  │     2824 │ 8534 │                 376 │                 0 │
└─────┴──────────┴──────┴─────────────────────┴───────────────────┘

entity-count delta between arms: 97 (conflates extraction variance with v2's higher defer rate - reported, not 
assumed away)

token-cap retries total: 0 (local engine - Bedrock daily cap escaped)

## Write report

In [9]:
report = {
    "hypothesis": "R15-H241",
    "stamp": STAMP,
    "design": "paired v1/v2 identity A/B, one local engine (gpt-oss-120b), two sequential scratch ingests, wipe between",
    "instance": FRESH_URI, "engine": "local vLLM gpt-oss-120b @ localhost:8010",
    "corpus": CORPUS, "n_docs": len(docs), "bench": BENCH, "probes": PROBES, "top_k": TOP_K,
    "bars": {"recall_delta_min": BAR_RECALL_DELTA, "precision_min": BAR_PRECISION,
             "false_merge_factor": FALSE_MERGE_FACTOR},
    "arms": {a: {"identity_stack": a, "config": m["config"],
                 "precision_proxy": results[a]["precision_proxy"],
                 "recall": results[a]["recall"],
                 "telemetry": tel[a]} for a, m in ARMS.items()},
    "clauses": {
        "non_regression": {"recall_delta": recall_delta, "bar": BAR_RECALL_DELTA, "pass": bool(nonreg_ok)},
        "false_merges": {"v1": p1["false_merges"], "v2": p2["false_merges"],
                          "ratio_fewer": fm_ratio, "pass": bool(fm_ok)},
        "precision_proxy": {"v2": p2["same_as_precision"], "bar": BAR_PRECISION, "pass": bool(prec_ok)},
    },
    "recommendation": "GO" if go else "NO-GO",
    "verdict_branch": ("CONFIRMED + flip approved" if go else
                       ("REFUTED - recall regression > 2pts" if refuted else "clause failure - flip not approved")),
    "confound": "temp-0 extraction variance is model-inherent (~0.60 JD, H229); two ingests do not share a byte-identical "
                "extraction stream; entity-count delta conflates extraction variance with v2 defer-rate; reported not assumed away",
}
Path(REPORT).write_text(json.dumps(report, indent=2, default=str))
rprint(f"[green]wrote[/green] {REPORT}")

wrote reports/identity-ab-h241-20260708T160009Z.json

## Verdict

Grounded in the clause table above. This cell's markdown is finalized by the executor after the run against the
printed outputs - the GO/NO-GO recommendation, which clauses passed, and the extraction-variance confound as
observed on the paired arms.